# GeoDiscoCats: Hypothesis Testing Notebook

This notebook demonstrates statistical hypothesis testing techniques using geographic cat discovery data.

**Compatible with:** Modal, Deepnote, Jupyter, Google Colab

## Overview

We'll explore various hypothesis tests including:
1. **One-Sample T-Test** - Testing if mean differs from expected value
2. **Two-Sample T-Test** - Comparing means between two groups
3. **Chi-Square Test** - Testing independence of categorical variables
4. **ANOVA** - Comparing means across multiple groups
5. **Correlation Test** - Testing relationship between variables

## Setup and Dependencies

In [ ]:
# Install dependencies if needed (uncomment for Modal/Deepnote)
# !pip install numpy pandas scipy matplotlib seaborn

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Dependencies loaded successfully!')

## Generate Sample Data

We'll create synthetic geographic cat discovery data for our hypothesis tests.

In [ ]:
# Generate synthetic GeoDiscoCats dataset
n_cats = 500

data = {
    'cat_id': range(1, n_cats + 1),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n_cats),
    'habitat': np.random.choice(['Urban', 'Rural', 'Suburban'], n_cats, p=[0.5, 0.2, 0.3]),
    'latitude': np.random.normal(40, 5, n_cats),
    'longitude': np.random.normal(-100, 10, n_cats),
    'weight_kg': np.random.normal(4.5, 1.2, n_cats),
    'age_years': np.random.exponential(5, n_cats),
    'discovery_hour': np.random.choice(range(24), n_cats),
    'is_friendly': np.random.choice([True, False], n_cats, p=[0.7, 0.3])
}

# Add region-specific weight variations
region_weight_adjustment = {'North': 0.5, 'South': -0.3, 'East': 0.2, 'West': -0.1}
data['weight_kg'] = [w + region_weight_adjustment[r] for w, r in zip(data['weight_kg'], data['region'])]

df = pd.DataFrame(data)
print(f'Dataset shape: {df.shape}')
df.head(10)

In [ ]:
# Dataset summary statistics
df.describe()

---

## Hypothesis Test 1: One-Sample T-Test

**Research Question:** Is the average weight of discovered cats different from the expected population mean of 4.5 kg?

- **H0 (Null Hypothesis):** The mean weight equals 4.5 kg
- **H1 (Alternative Hypothesis):** The mean weight differs from 4.5 kg

In [ ]:
# One-Sample T-Test
expected_weight = 4.5
sample_weights = df['weight_kg']

t_statistic, p_value = stats.ttest_1samp(sample_weights, expected_weight)

print('=' * 50)
print('ONE-SAMPLE T-TEST RESULTS')
print('=' * 50)
print(f'Sample Mean: {sample_weights.mean():.3f} kg')
print(f'Expected Mean: {expected_weight} kg')
print(f'Sample Std Dev: {sample_weights.std():.3f} kg')
print(f'T-Statistic: {t_statistic:.4f}')
print(f'P-Value: {p_value:.4f}')
print('-' * 50)

alpha = 0.05
if p_value < alpha:
    print(f'Result: REJECT H0 at alpha={alpha}')
    print('Conclusion: Mean weight significantly differs from 4.5 kg')
else:
    print(f'Result: FAIL TO REJECT H0 at alpha={alpha}')
    print('Conclusion: No significant difference from 4.5 kg')

In [ ]:
# Visualize the distribution
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(sample_weights, kde=True, ax=ax, color='steelblue')
ax.axvline(expected_weight, color='red', linestyle='--', linewidth=2, label=f'Expected ({expected_weight} kg)')
ax.axvline(sample_weights.mean(), color='green', linestyle='-', linewidth=2, label=f'Sample Mean ({sample_weights.mean():.2f} kg)')
ax.set_xlabel('Weight (kg)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Cat Weights')
ax.legend()
plt.tight_layout()
plt.show()

---

## Hypothesis Test 2: Two-Sample T-Test (Independent)

**Research Question:** Do cats in Urban vs Rural habitats have different weights?

- **H0:** Mean weight of Urban cats = Mean weight of Rural cats
- **H1:** Mean weight of Urban cats ≠ Mean weight of Rural cats

In [ ]:
# Two-Sample Independent T-Test
urban_weights = df[df['habitat'] == 'Urban']['weight_kg']
rural_weights = df[df['habitat'] == 'Rural']['weight_kg']

t_stat, p_val = stats.ttest_ind(urban_weights, rural_weights)

print('=' * 50)
print('TWO-SAMPLE INDEPENDENT T-TEST RESULTS')
print('=' * 50)
print(f'Urban cats (n={len(urban_weights)}): Mean = {urban_weights.mean():.3f} kg')
print(f'Rural cats (n={len(rural_weights)}): Mean = {rural_weights.mean():.3f} kg')
print(f'Difference: {urban_weights.mean() - rural_weights.mean():.3f} kg')
print(f'T-Statistic: {t_stat:.4f}')
print(f'P-Value: {p_val:.4f}')
print('-' * 50)

if p_val < alpha:
    print(f'Result: REJECT H0 at alpha={alpha}')
    print('Conclusion: Significant weight difference between habitats')
else:
    print(f'Result: FAIL TO REJECT H0 at alpha={alpha}')
    print('Conclusion: No significant weight difference between habitats')

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(8, 6))
habitat_data = df[df['habitat'].isin(['Urban', 'Rural'])]
sns.boxplot(x='habitat', y='weight_kg', data=habitat_data, ax=ax)
ax.set_xlabel('Habitat Type')
ax.set_ylabel('Weight (kg)')
ax.set_title('Cat Weight by Habitat Type')
plt.tight_layout()
plt.show()

---

## Hypothesis Test 3: Chi-Square Test of Independence

**Research Question:** Is there an association between habitat type and cat friendliness?

- **H0:** Habitat and friendliness are independent
- **H1:** Habitat and friendliness are not independent

In [ ]:
# Chi-Square Test of Independence
contingency_table = pd.crosstab(df['habitat'], df['is_friendly'])
print('Contingency Table:')
print(contingency_table)
print()

chi2, p_val, dof, expected_freq = stats.chi2_contingency(contingency_table)

print('=' * 50)
print('CHI-SQUARE TEST OF INDEPENDENCE RESULTS')
print('=' * 50)
print(f'Chi-Square Statistic: {chi2:.4f}')
print(f'Degrees of Freedom: {dof}')
print(f'P-Value: {p_val:.4f}')
print('-' * 50)

if p_val < alpha:
    print(f'Result: REJECT H0 at alpha={alpha}')
    print('Conclusion: Habitat and friendliness are NOT independent')
else:
    print(f'Result: FAIL TO REJECT H0 at alpha={alpha}')
    print('Conclusion: No significant association between habitat and friendliness')

In [ ]:
# Visualize the contingency table
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Observed frequencies
sns.heatmap(contingency_table, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Observed Frequencies')

# Expected frequencies
expected_df = pd.DataFrame(expected_freq, index=contingency_table.index, columns=contingency_table.columns)
sns.heatmap(expected_df, annot=True, fmt='.1f', cmap='Greens', ax=axes[1])
axes[1].set_title('Expected Frequencies (under H0)')

plt.tight_layout()
plt.show()

---

## Hypothesis Test 4: One-Way ANOVA

**Research Question:** Are there differences in cat weights across the four geographic regions?

- **H0:** Mean weights are equal across all regions (μ_North = μ_South = μ_East = μ_West)
- **H1:** At least one region has a different mean weight

In [ ]:
# One-Way ANOVA
north_weights = df[df['region'] == 'North']['weight_kg']
south_weights = df[df['region'] == 'South']['weight_kg']
east_weights = df[df['region'] == 'East']['weight_kg']
west_weights = df[df['region'] == 'West']['weight_kg']

f_stat, p_val = stats.f_oneway(north_weights, south_weights, east_weights, west_weights)

print('=' * 50)
print('ONE-WAY ANOVA RESULTS')
print('=' * 50)
print('\nGroup Statistics:')
for region in ['North', 'South', 'East', 'West']:
    region_data = df[df['region'] == region]['weight_kg']
    print(f'  {region}: n={len(region_data)}, Mean={region_data.mean():.3f}, Std={region_data.std():.3f}')

print(f'\nF-Statistic: {f_stat:.4f}')
print(f'P-Value: {p_val:.6f}')
print('-' * 50)

if p_val < alpha:
    print(f'Result: REJECT H0 at alpha={alpha}')
    print('Conclusion: Significant weight differences exist between regions')
else:
    print(f'Result: FAIL TO REJECT H0 at alpha={alpha}')
    print('Conclusion: No significant weight differences between regions')

In [ ]:
# Post-hoc analysis: Tukey's HSD (if ANOVA is significant)
from scipy.stats import tukey_hsd

if p_val < alpha:
    print('Post-hoc Tukey HSD Test:')
    print('-' * 50)
    
    result = tukey_hsd(north_weights, south_weights, east_weights, west_weights)
    regions = ['North', 'South', 'East', 'West']
    
    print('\nPairwise comparisons (p-values):')
    for i in range(4):
        for j in range(i+1, 4):
            pval = result.pvalue[i, j]
            sig = '*' if pval < 0.05 else ''
            print(f'  {regions[i]} vs {regions[j]}: p = {pval:.4f} {sig}')

In [ ]:
# Visualize ANOVA
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
sns.boxplot(x='region', y='weight_kg', data=df, ax=axes[0], order=['North', 'South', 'East', 'West'])
axes[0].set_xlabel('Region')
axes[0].set_ylabel('Weight (kg)')
axes[0].set_title('Cat Weight by Geographic Region')

# Violin plot
sns.violinplot(x='region', y='weight_kg', data=df, ax=axes[1], order=['North', 'South', 'East', 'West'])
axes[1].set_xlabel('Region')
axes[1].set_ylabel('Weight (kg)')
axes[1].set_title('Weight Distribution by Region')

plt.tight_layout()
plt.show()

---

## Hypothesis Test 5: Pearson Correlation Test

**Research Question:** Is there a significant correlation between cat age and weight?

- **H0:** No linear correlation exists (ρ = 0)
- **H1:** A linear correlation exists (ρ ≠ 0)

In [ ]:
# Pearson Correlation Test
correlation, p_val = stats.pearsonr(df['age_years'], df['weight_kg'])

print('=' * 50)
print('PEARSON CORRELATION TEST RESULTS')
print('=' * 50)
print(f'Pearson Correlation Coefficient (r): {correlation:.4f}')
print(f'P-Value: {p_val:.4f}')
print('-' * 50)

# Interpret correlation strength
r_abs = abs(correlation)
if r_abs < 0.1:
    strength = 'negligible'
elif r_abs < 0.3:
    strength = 'weak'
elif r_abs < 0.5:
    strength = 'moderate'
elif r_abs < 0.7:
    strength = 'strong'
else:
    strength = 'very strong'

print(f'Correlation Strength: {strength}')

if p_val < alpha:
    print(f'Result: REJECT H0 at alpha={alpha}')
    print('Conclusion: Significant correlation exists between age and weight')
else:
    print(f'Result: FAIL TO REJECT H0 at alpha={alpha}')
    print('Conclusion: No significant correlation between age and weight')

In [ ]:
# Visualize correlation
fig, ax = plt.subplots(figsize=(10, 6))
sns.regplot(x='age_years', y='weight_kg', data=df, ax=ax, scatter_kws={'alpha': 0.5})
ax.set_xlabel('Age (years)')
ax.set_ylabel('Weight (kg)')
ax.set_title(f'Age vs Weight Correlation (r = {correlation:.3f}, p = {p_val:.4f})')
plt.tight_layout()
plt.show()

---

## Summary of All Tests

In [ ]:
# Create summary table
summary_data = {
    'Test': [
        'One-Sample T-Test',
        'Two-Sample T-Test',
        'Chi-Square Test',
        'One-Way ANOVA',
        'Pearson Correlation'
    ],
    'Research Question': [
        'Mean weight = 4.5 kg?',
        'Urban vs Rural weights differ?',
        'Habitat & friendliness independent?',
        'Weight differs across regions?',
        'Age-weight correlation exists?'
    ],
    'Test Statistic': [
        f't = {stats.ttest_1samp(df["weight_kg"], 4.5)[0]:.3f}',
        f't = {stats.ttest_ind(urban_weights, rural_weights)[0]:.3f}',
        f'χ² = {chi2:.3f}',
        f'F = {f_stat:.3f}',
        f'r = {correlation:.3f}'
    ],
    'P-Value': [
        f'{stats.ttest_1samp(df["weight_kg"], 4.5)[1]:.4f}',
        f'{stats.ttest_ind(urban_weights, rural_weights)[1]:.4f}',
        f'{stats.chi2_contingency(contingency_table)[1]:.4f}',
        f'{p_val:.6f}',
        f'{stats.pearsonr(df["age_years"], df["weight_kg"])[1]:.4f}'
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_df['Significant (α=0.05)'] = summary_df['P-Value'].apply(
    lambda x: 'Yes' if float(x) < 0.05 else 'No'
)

print('=' * 80)
print('HYPOTHESIS TESTS SUMMARY')
print('=' * 80)
print(summary_df.to_string(index=False))

---

## Additional Resources

### Interpreting Results

| P-Value Range | Interpretation |
|--------------|----------------|
| p < 0.001 | Very strong evidence against H0 |
| 0.001 ≤ p < 0.01 | Strong evidence against H0 |
| 0.01 ≤ p < 0.05 | Moderate evidence against H0 |
| 0.05 ≤ p < 0.1 | Weak evidence against H0 |
| p ≥ 0.1 | Little to no evidence against H0 |

### Assumptions to Check

- **T-Tests**: Normality (for small samples), equal variances (for independent t-test)
- **Chi-Square**: Expected frequencies ≥ 5 in each cell
- **ANOVA**: Normality within groups, homogeneity of variances
- **Pearson Correlation**: Linear relationship, normality of both variables

---

*Notebook created for GeoDiscoCats hypothesis testing demonstration.*